# Abundant-data graph-regularization sweep — Gini refinement

This is a **focused tuning/ablation experiment** for the finalized
unbounded-\(\alpha\), \(\lambda_{\rm mix}=1\) identifier.

The motivation is the observed abundant-data failure mode in which the learned
weighted adjacency matrix \(\hat A\) becomes diffuse even though substantially
more transition data are available.

We compare four conditions:

1. **None** — the exact finalized September model, loaded from its existing cache.
2. **Paper \(L_2(\Theta)\)** — literal paper-inspired penalty
   \(\lambda\sum_{ij}\Theta_{ij}^2\).
3. **Entropy** — minimize normalized row entropy of \(\hat A\).
4. **Gini impurity** — minimize normalized row Gini impurity of \(\hat A\).

## Experimental isolation

Every new candidate:

- uses the exact same 195 final benchmark environments;
- uses the exact same cached **2500 abundant training pairs per environment**;
- uses learning seed 0 for this coarse sweep;
- uses the exact same architecture, optimizer, training budget and frozen-control
  evaluation as the final September abundant experiment;
- changes **only** the graph regularizer.

No environment data are recollected.

This is a **coarse tuning sweep**, not the final multi-seed validation. After we
identify a promising family/\(\lambda\), that candidate should be validated with
all three learning seeds.

### Final Gini refinement

This pass reuses all previous entropy and theta-L2 conditions and the original Gini points. It computes only $\lambda_G\in\{3\times10^{-5},5\times10^{-5},2\times10^{-4},3\times10^{-4},5\times10^{-4}\}$.


In [ ]:
from __future__ import annotations

import contextlib
import hashlib
import inspect
import io
import json
import math
import os
import random
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 300)
pd.set_option("display.max_rows", 300)


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "opinion_dynamics").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing opinion_dynamics/.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from rl_envs_forge.envs.network_graph.network_graph import NetworkGraph
from rl_envs_forge.envs.network_graph.graph_utils import (
    compute_eigenvector_centrality,
    compute_laplacian,
)
from opinion_dynamics.identify_nonlinear_unbounded_regularized import (
    GraphIdentifierEnv,
    train_graph_identifier,
)
import opinion_dynamics.experiments.online_single_shot_unbounded as online_single_shot

try:
    GIT_COMMIT = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_ROOT,
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
except Exception:
    GIT_COMMIT = "unknown"

print("Repository:", REPO_ROOT)
print("Git commit:", GIT_COMMIT)
print("Identifier:", inspect.getsourcefile(GraphIdentifierEnv))


## Frozen final configuration and sweep grid

In [ ]:
NUM_SHARDS_ENV = "REG_SWEEP_NUM_SHARDS"
NUM_SHARDS = int(os.environ.get(NUM_SHARDS_ENV, "8"))
if NUM_SHARDS < 1:
    raise ValueError(f"{NUM_SHARDS_ENV} must be >= 1; got {NUM_SHARDS}")
SHARD_ENV = "REG_SWEEP_SHARD_ID"
THREAD_ENV = "REG_SWEEP_TORCH_THREADS"
_shard_value = os.environ.get(SHARD_ENV)
WORKER_MODE = _shard_value is not None
SHARD_ID = int(_shard_value) if WORKER_MODE else None
if WORKER_MODE and SHARD_ID not in range(NUM_SHARDS):
    raise ValueError(f"{SHARD_ENV} must be in 0..{NUM_SHARDS-1}")

TORCH_THREADS = int(os.environ.get(THREAD_ENV, "2"))
torch.set_num_threads(max(1, TORCH_THREADS))
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

DEVICE = "cpu"

# Final September settings.
N = 15
TARGET = 1.0
NUM_CAMPAIGNS = 20
T_CAMPAIGN = 0.5
T_S = 0.1
MAX_U = 0.20
TOTAL_CONTROLLED_BUDGET = 6.0
B_CAMPAIGN = TOTAL_CONTROLLED_BUDGET / (NUM_CAMPAIGNS - 1)
LAMBDA_MIX = 1.0

FIT_LR = 1e-3
FIT_MAX_STEPS = 1_000
FIT_MAE_STOP = 5e-4
FIT_BATCH_SIZE = 256
FIT_CHECK_EVERY = 200
IDENTIFIER_KWARGS = {"hidden_dim": 16}

# Coarse tuning uses one fixed optimization seed.
LEARNING_SEED = 0

# Exact source abundant run whose 2500-pair datasets are reused.
SOURCE_STUDY = "abundant_all_dynamics_multitopology_multiseed_unbounded_lambda1"
SOURCE_CONFIG_HASH = "2dfd8a85437a16ee"
SOURCE_PIPELINE_VERSION = "2026-09-17-unbounded-lambda1-v1"

# Final focused Gini refinement.
# Existing points at 1e-6, 1e-5, 1e-4 and 1e-3 are reused from the
# original sweep. We add five points in the transition region where the
# previous results showed the best recovery/control trade-off.
#
# Theta-L2 and entropy are retained in the combined analysis and reused
# entirely from their previous caches; they are not retrained here.
SWEEP = {
    "theta_l2": [1e-8, 1e-7, 1e-6, 1e-5, 1e-4],
    "entropy":  [1e-6, 1e-5, 1e-4, 1e-3],
    "gini":     [1e-6, 1e-5, 3e-5, 5e-5, 1e-4, 2e-4, 3e-4, 5e-4, 1e-3],
}

FIRST_SWEEP_CONFIG_HASH = "372b37a37b77b2f9"
THETA_REFINEMENT_CONFIG_HASH = "75d7a9c314c10b65"

Gini_NEW_LAMBDAS = {3e-5, 5e-5, 2e-4, 3e-4, 5e-4}
FIRST_SWEEP_GINI_LAMBDAS = {1e-6, 1e-5, 1e-4, 1e-3}

VARIANTS = [
    {"regularizer_kind": kind, "regularizer_lambda": float(lam)}
    for kind, lambdas in SWEEP.items()
    for lam in lambdas
]

STUDY_NAME = "abundant_regularization_sweep_unbounded_lambda1"
PIPELINE_VERSION = "2026-09-27-regularization-gini-refinement-v3"

RESULTS_ROOT = REPO_ROOT / "opinion_dynamics" / "experiments" / "results"
SOURCE_RESULTS_DIR = RESULTS_ROOT / "experiment_2026_09_17_abundant_all_dynamics_multitopology_multiseed_unbounded_lambda1"
SOURCE_CACHE_ROOT = RESULTS_ROOT / "_trial_cache" / SOURCE_STUDY / SOURCE_CONFIG_HASH

RESULTS_DIR = RESULTS_ROOT / "experiment_2026_09_25_abundant_regularization_sweep"
CACHE_ROOT = RESULTS_ROOT / "_trial_cache" / STUDY_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print("Mode:", "worker" if WORKER_MODE else "merge/analysis")
print("Configured shards:", NUM_SHARDS)
if WORKER_MODE:
    print("Shard:", f"{SHARD_ID + 1}/{NUM_SHARDS}")
print("Regularized variants in combined analysis:", len(VARIANTS))
print("New Gini fits expected:", 195 * len(Gini_NEW_LAMBDAS))
print("Reused theta_l2 fits:", 195 * len(SWEEP["theta_l2"]))
print("Reused entropy fits:", 195 * len(SWEEP["entropy"]))
print("Reused original Gini fits:", 195 * len(FIRST_SWEEP_GINI_LAMBDAS))
print("Source dataset cache:", SOURCE_CACHE_ROOT)


## Exact final benchmark graph constructors

In [ ]:
UNSTRUCTURED_TOPOLOGY_SEEDS = [3, 4, 5, 6, 7]
UNSTRUCTURED_OPINION_SEEDS = [0, 1, 2, 4, 5]
UNSTRUCTURED_DYNAMICS = {
    "laplacian": {},
    "coca": {},
    "hegselmannkrause": {"hk_epsilon": 0.50, "hk_include_self": True},
    "friedkinjohnsen": {"fj_lambda": 0.98},
    "nonlinearinfluence": {"nonlinear_beta": 4.0},
    "repulsion": {"repulsion_epsilon": 0.30, "repulsion_strength": 0.10},
}

STRUCTURED_HK_EPSILON = 0.25
STRUCTURED_FJ_LAMBDA = 0.98
STRUCTURED_NLI_BETA = 4.0
STRUCTURED_ENVIRONMENT_SEEDS = [0, 1, 2, 3, 4]

HK_FAMILIES = {
    "ring_bridge": {"peer_pattern":"ring","hub_leaf_weight":3.0,"peer_weight":0.40,"bridge_weight":3.0},
    "paired_bridge": {"peer_pattern":"paired","hub_leaf_weight":3.0,"peer_weight":0.55,"bridge_weight":3.0},
    "dense_weak_bridge": {"peer_pattern":"dense","hub_leaf_weight":3.0,"peer_weight":0.08,"bridge_weight":3.0},
}
BROADCAST_FAMILIES = {
    "asymmetric_broadcast": {"kind":"asymmetric","hub_influence_total":0.85},
    "dual_broadcast": {"kind":"dual","hub_influence_total":0.85},
    "split_broadcast": {"kind":"split","hub_influence_total":0.85},
}
FJ_HUB_LEVEL = 0.16
FJ_LEAF_LEVEL = 0.48
NLI_HUB_LEVEL = 0.10
NLI_LEAF_LEVEL = 0.50
EDGE_WEIGHT_JITTER_FRAC = 0.05
HK_OPINION_JITTER = 0.006
BROADCAST_OPINION_JITTER = 0.010


def _normalize_rows(raw):
    raw = np.asarray(raw, dtype=float).copy()
    np.fill_diagonal(raw, 0.0)
    sums = raw.sum(axis=1, keepdims=True)
    if np.any(sums <= 0):
        raise RuntimeError("Every graph row must have positive mass.")
    A = raw / sums
    np.fill_diagonal(A, 0.0)
    return A


def _jitter_positive_edges(raw, rng, frac):
    out = np.asarray(raw, dtype=float).copy()
    mask = out > 0
    multipliers = rng.uniform(1.0-frac, 1.0+frac, size=out.shape)
    out[mask] *= multipliers[mask]
    np.fill_diagonal(out, 0.0)
    return out


def _add_undirected(raw, i, j, weight):
    raw[i,j] += float(weight)
    raw[j,i] += float(weight)


def centrality_from_A(A):
    v = compute_eigenvector_centrality(compute_laplacian(np.asarray(A, dtype=float)))
    v = np.asarray(v, dtype=float).reshape(-1)
    v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)
    if v.sum() < 0:
        v = -v
    v = np.maximum(v, 0.0)
    if v.sum() <= 1e-12:
        v = np.abs(v)
    if v.sum() <= 1e-12:
        return np.full(N, 1.0/N)
    return v / v.sum()


def build_unstructured_adjacency(topology_seed):
    x_dummy = np.linspace(0.0, 1.0, N)
    graph_env = NetworkGraph(
        num_agents=N, graph_model="barabasi_albert", ba_m=2,
        ba_prune_max_frac=0.5, ba_qsc_tol=1e-8, ba_max_tries=500,
        max_u=MAX_U, budget=1000.0, desired_opinion=TARGET,
        t_campaign=T_CAMPAIGN, t_s=T_S, max_steps=NUM_CAMPAIGNS+20,
        opinion_end_tolerance=0.01, control_beta=0.4,
        normalize_reward=True, terminal_reward=0.0,
        seed=int(topology_seed), terminate_when_converged=False,
        dynamics_model="laplacian", initial_opinions=x_dummy,
        control_resistance=np.zeros(N, dtype=float),
    )
    return np.asarray(graph_env.connectivity_matrix, dtype=float).copy()


def build_unstructured_x0(opinion_seed):
    rng = np.random.default_rng(920_000 + int(opinion_seed))
    return rng.permutation(np.linspace(0.0, 1.0, N)).astype(float)


UNSTRUCTURED_A_BY_SEED = {
    int(seed): build_unstructured_adjacency(int(seed))
    for seed in UNSTRUCTURED_TOPOLOGY_SEEDS
}

HK_LOW_CLUSTER = tuple(range(0,7))
HK_HIGH_CLUSTER = tuple(range(7,15))
HK_LOW_HUB = 0
HK_HIGH_HUB = 7
BROADCAST_HUBS = (0,1)
BROADCAST_LEAVES = tuple(range(2,N))
BROADCAST_GROUP_A = tuple(range(2,8))
BROADCAST_GROUP_B = tuple(range(8,N))


def build_hk_graph(family_name, environment_seed):
    params = HK_FAMILIES[family_name]
    raw = np.zeros((N,N), dtype=float)

    for node in HK_LOW_CLUSTER:
        if node != HK_LOW_HUB:
            _add_undirected(raw, HK_LOW_HUB, node, params["hub_leaf_weight"])
    for node in HK_HIGH_CLUSTER:
        if node != HK_HIGH_HUB:
            _add_undirected(raw, HK_HIGH_HUB, node, params["hub_leaf_weight"])

    for cluster, hub in [(HK_LOW_CLUSTER,HK_LOW_HUB),(HK_HIGH_CLUSTER,HK_HIGH_HUB)]:
        leaves = [node for node in cluster if node != hub]
        pattern = params["peer_pattern"]
        w = float(params["peer_weight"])
        if pattern == "ring":
            for left, right in zip(leaves, leaves[1:]+leaves[:1]):
                _add_undirected(raw,left,right,w)
        elif pattern == "paired":
            for index in range(0,len(leaves)-1,2):
                _add_undirected(raw,leaves[index],leaves[index+1],w)
            if len(leaves)%2 == 1:
                _add_undirected(raw,leaves[-1],leaves[0],0.5*w)
        elif pattern == "dense":
            for pos,left in enumerate(leaves):
                for right in leaves[pos+1:]:
                    _add_undirected(raw,left,right,w)
        else:
            raise ValueError(pattern)

    _add_undirected(raw,HK_LOW_HUB,HK_HIGH_HUB,params["bridge_weight"])
    rng = np.random.default_rng(
        10_000 + 100*list(HK_FAMILIES).index(family_name) + int(environment_seed)
    )
    raw = _jitter_positive_edges(raw,rng,EDGE_WEIGHT_JITTER_FRAC)
    return raw, _normalize_rows(raw)


def build_hk_x0(environment_seed):
    base = np.array(
        [0.35,0.27,0.28,0.29,0.30,0.31,0.32,
         0.64,0.65,0.66,0.67,0.68,0.69,0.70,0.71],
        dtype=float,
    )
    rng = np.random.default_rng(20_000 + int(environment_seed))
    return np.clip(
        base + rng.uniform(-HK_OPINION_JITTER,HK_OPINION_JITTER,size=N),
        0.0,1.0,
    )


def build_broadcaster_graph(family_name, environment_seed):
    params = BROADCAST_FAMILIES[family_name]
    h = float(params["hub_influence_total"])
    kind = params["kind"]
    raw = np.zeros((N,N), dtype=float)
    leaves = list(BROADCAST_LEAVES)

    for idx,leaf in enumerate(leaves):
        prev_leaf = leaves[(idx-1)%len(leaves)]
        next_leaf = leaves[(idx+1)%len(leaves)]
        if kind == "dual":
            raw[leaf,0] = 0.50*h; raw[leaf,1] = 0.50*h
        elif kind == "split":
            primary = 0 if leaf in BROADCAST_GROUP_A else 1
            secondary = 1 if primary == 0 else 0
            raw[leaf,primary] = 0.85*h; raw[leaf,secondary] = 0.15*h
        elif kind == "asymmetric":
            raw[leaf,0] = 0.75*h; raw[leaf,1] = 0.25*h
        else:
            raise ValueError(kind)
        remaining = 1.0-h
        raw[leaf,prev_leaf] += 0.50*remaining
        raw[leaf,next_leaf] += 0.50*remaining

    raw[0,1] = 0.20
    raw[1,0] = 0.20
    for leaf in BROADCAST_GROUP_A:
        raw[0,leaf] += 0.80/len(BROADCAST_GROUP_A)
    for leaf in BROADCAST_GROUP_B:
        raw[1,leaf] += 0.80/len(BROADCAST_GROUP_B)

    rng = np.random.default_rng(
        30_000 + 100*list(BROADCAST_FAMILIES).index(family_name) + int(environment_seed)
    )
    raw = _jitter_positive_edges(raw,rng,EDGE_WEIGHT_JITTER_FRAC)
    return raw, _normalize_rows(raw)


def build_broadcast_x0(*, hub_level, leaf_level, environment_seed):
    x0 = np.empty(N,dtype=float)
    x0[0] = float(hub_level)-0.01
    x0[1] = float(hub_level)+0.01
    x0[list(BROADCAST_LEAVES)] = (
        float(leaf_level) + np.linspace(-0.035,0.035,len(BROADCAST_LEAVES))
    )
    rng = np.random.default_rng(40_000 + int(environment_seed))
    x0 += rng.uniform(-BROADCAST_OPINION_JITTER,BROADCAST_OPINION_JITTER,size=N)
    return np.clip(x0,0.0,1.0)


## Build the identical 195-environment registry

In [ ]:
ENVIRONMENT_SPECS = []

for dynamics, dynamics_params in UNSTRUCTURED_DYNAMICS.items():
    for topology_seed in UNSTRUCTURED_TOPOLOGY_SEEDS:
        for opinion_seed in UNSTRUCTURED_OPINION_SEEDS:
            A = UNSTRUCTURED_A_BY_SEED[int(topology_seed)].copy()
            x0 = build_unstructured_x0(int(opinion_seed))
            ENVIRONMENT_SPECS.append({
                "environment_index": len(ENVIRONMENT_SPECS),
                "environment_id": f"unstructured__{dynamics}__topo{topology_seed:02d}__x{opinion_seed:02d}",
                "scenario_class": "unstructured_random",
                "family": "barabasi_albert_spread",
                "dynamics": dynamics,
                "topology_seed": int(topology_seed),
                "opinion_seed": int(opinion_seed),
                "environment_seed": None,
                "dynamics_params": dict(dynamics_params),
                "A": A,
                "x0": x0,
                "v_true": centrality_from_A(A),
            })

for family in HK_FAMILIES:
    for environment_seed in STRUCTURED_ENVIRONMENT_SEEDS:
        raw,A = build_hk_graph(family,environment_seed)
        x0 = build_hk_x0(environment_seed)
        ENVIRONMENT_SPECS.append({
            "environment_index": len(ENVIRONMENT_SPECS),
            "environment_id": f"structured__hegselmannkrause__{family}__env{environment_seed:02d}",
            "scenario_class": "structured_mechanism",
            "family": family,
            "dynamics": "hegselmannkrause",
            "topology_seed": None,
            "opinion_seed": None,
            "environment_seed": int(environment_seed),
            "dynamics_params": {"hk_epsilon":STRUCTURED_HK_EPSILON,"hk_include_self":True},
            "A": A, "x0": x0, "v_true": centrality_from_A(A),
        })

for dynamics in ["friedkinjohnsen","nonlinearinfluence"]:
    for family in BROADCAST_FAMILIES:
        for environment_seed in STRUCTURED_ENVIRONMENT_SEEDS:
            raw,A = build_broadcaster_graph(family,environment_seed)
            if dynamics == "friedkinjohnsen":
                x0 = build_broadcast_x0(
                    hub_level=FJ_HUB_LEVEL,leaf_level=FJ_LEAF_LEVEL,
                    environment_seed=environment_seed,
                )
                dynamics_params = {"fj_lambda":STRUCTURED_FJ_LAMBDA}
            else:
                x0 = build_broadcast_x0(
                    hub_level=NLI_HUB_LEVEL,leaf_level=NLI_LEAF_LEVEL,
                    environment_seed=environment_seed,
                )
                dynamics_params = {"nonlinear_beta":STRUCTURED_NLI_BETA}
            ENVIRONMENT_SPECS.append({
                "environment_index": len(ENVIRONMENT_SPECS),
                "environment_id": f"structured__{dynamics}__{family}__env{environment_seed:02d}",
                "scenario_class": "structured_mechanism",
                "family": family,
                "dynamics": dynamics,
                "topology_seed": None,
                "opinion_seed": None,
                "environment_seed": int(environment_seed),
                "dynamics_params": dynamics_params,
                "A": A, "x0": x0, "v_true": centrality_from_A(A),
            })

assert len(ENVIRONMENT_SPECS) == 195
print(pd.DataFrame(ENVIRONMENT_SPECS)[["scenario_class","dynamics"]].value_counts().sort_index())


## Dataset reuse, model fitting, frozen-control evaluation, and recovery metrics

In [ ]:
def make_env(spec, *, seed):
    params = dict(spec["dynamics_params"])
    kwargs = dict(
        connectivity_matrix=np.asarray(spec["A"],dtype=float).copy(),
        num_agents=N,
        max_u=MAX_U,
        desired_opinion=TARGET,
        t_campaign=T_CAMPAIGN,
        t_s=T_S,
        initial_opinions=np.asarray(spec["x0"],dtype=float).copy(),
        control_resistance=np.zeros(N,dtype=float),
        dynamics_model=spec["dynamics"],
        budget=1000.0,
        max_steps=NUM_CAMPAIGNS+20,
        opinion_end_tolerance=0.01,
        control_beta=0.4,
        normalize_reward=True,
        terminal_reward=0.0,
        terminate_when_converged=False,
        seed=int(seed),
    )
    if spec["dynamics"] == "hegselmannkrause":
        kwargs["hk_epsilon"] = float(params["hk_epsilon"])
        kwargs["hk_include_self"] = bool(params["hk_include_self"])
    elif spec["dynamics"] == "friedkinjohnsen":
        kwargs["fj_lambda"] = float(params["fj_lambda"])
        kwargs["fj_prejudice"] = np.asarray(spec["x0"],dtype=float).copy()
    elif spec["dynamics"] == "nonlinearinfluence":
        kwargs["nonlinear_beta"] = float(params["nonlinear_beta"])
    elif spec["dynamics"] == "repulsion":
        kwargs["repulsion_epsilon"] = float(params["repulsion_epsilon"])
        kwargs["repulsion_strength"] = float(params["repulsion_strength"])
    return NetworkGraph(**kwargs)


def set_state(env, x0):
    env.reset()
    env.opinions = np.asarray(x0,dtype=float).copy()
    if hasattr(env,"state"):
        try:
            env.state = np.asarray(x0,dtype=float).copy()
        except Exception:
            pass


def source_dataset_path(spec):
    return SOURCE_CACHE_ROOT / spec["environment_id"] / "dataset.npz"


def source_dataset_meta_path(spec):
    return SOURCE_CACHE_ROOT / spec["environment_id"] / "dataset_meta.json"


def load_source_dataset(spec):
    dp = source_dataset_path(spec)
    mp = source_dataset_meta_path(spec)
    if not dp.exists() or not mp.exists():
        raise FileNotFoundError(
            f"Missing finalized abundant dataset for {spec['environment_id']}: {dp}"
        )
    meta = json.loads(mp.read_text(encoding="utf-8"))
    if meta.get("config_hash") != SOURCE_CONFIG_HASH:
        raise RuntimeError(
            f"Source dataset config mismatch for {spec['environment_id']}: "
            f"{meta.get('config_hash')} != {SOURCE_CONFIG_HASH}"
        )
    with np.load(dp) as arr:
        X = np.asarray(arr["X"],dtype=float)
        Y = np.asarray(arr["Y"],dtype=float)
    if len(X) != 2500:
        raise RuntimeError(f"{spec['environment_id']}: expected 2500 pairs, got {len(X)}")
    return X,Y,meta


def seed_everything(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))


def fit_variant(X,Y,*,kind,lam):
    seed_everything(LEARNING_SEED)
    with contextlib.redirect_stdout(io.StringIO()):
        model = GraphIdentifierEnv(
            N=N, s=T_S, zero_diag=True,
            regularizer_kind=kind,
            regularizer_lambda=float(lam),
            **IDENTIFIER_KWARGS,
        )
    t0=time.perf_counter()
    with contextlib.redirect_stdout(io.StringIO()):
        A_hat = train_graph_identifier(
            model,X,Y,lr=FIT_LR,batch_size=FIT_BATCH_SIZE,
            max_steps=FIT_MAX_STEPS,mae_stop=FIT_MAE_STOP,
            device=DEVICE,fit_check_every=FIT_CHECK_EVERY,verbose_every=0,
        )
    elapsed=time.perf_counter()-t0
    model.eval()
    with torch.no_grad():
        Xt=torch.tensor(X,dtype=torch.float32)
        Yt=torch.tensor(Y,dtype=torch.float32)
        pred=model.predict_next(Xt)
        train_mae=float((pred-Yt).abs().mean().item())
        identity_mae=float((Xt-Yt).abs().mean().item())
        reg_value=float(model.regularizer_value().item())
        A=np.asarray(model.A_hat().cpu().numpy(),dtype=float)
    return model,A,{
        "fit_elapsed_s":elapsed,
        "train_mae":train_mae,
        "identity_mae":identity_mae,
        "model_over_identity":train_mae/identity_mae if identity_mae>1e-12 else np.nan,
        "reg_value_final":reg_value,
        "weighted_reg_final":float(lam)*reg_value,
        "fit_steps_run":int(model.last_fit_info["steps_run"]),
        "fit_stop_reason":model.last_fit_info["stop_reason"],
    }


def rollout_model(spec,model):
    env=make_env(spec,seed=700_000+int(spec["environment_index"]))
    set_state(env,spec["x0"])
    state=np.asarray(spec["x0"],dtype=float).copy()
    states=[state.copy()]
    max_u_vec=np.asarray(env.max_u,dtype=float).reshape(-1)
    for campaign in range(NUM_CAMPAIGNS):
        if campaign==0:
            action=np.zeros(N,dtype=float)
        else:
            scores,_,_=online_single_shot.learned_lambda_mix_scores(
                model,state,desired_opinion=TARGET,
                lambda_mix=LAMBDA_MIX,device=DEVICE,
            )
            action=online_single_shot.waterfill_from_scores(
                scores,max_u=max_u_vec,budget=float(B_CAMPAIGN),
            )
        x_next,_,done,trunc,_=env.step(action)
        state=np.asarray(x_next,dtype=float).copy()
        states.append(state.copy())
        if done or trunc:
            break
    states=np.asarray(states,dtype=float)
    mean_path=states.mean(axis=1)
    return {
        "mean_end":float(mean_path[-1]),
        "mean_over_trajectory":float(mean_path.mean()),
        "final_mean_abs_distance_to_target":float(np.mean(np.abs(TARGET-states[-1]))),
    }


def edge_average_precision(A_true,A_hat,tol=1e-12):
    n=A_true.shape[0]
    mask=~np.eye(n,dtype=bool)
    y=(A_true[mask]>tol).astype(int)
    scores=A_hat[mask]
    order=np.argsort(-scores,kind="mergesort")
    ys=y[order]
    tp=np.cumsum(ys)
    precision=tp/(np.arange(len(ys))+1)
    return float(np.sum(precision*ys)/max(1,ys.sum()))


def support_topdegree_recall(A_true,A_hat,tol=1e-12):
    vals=[]
    for i in range(A_true.shape[0]):
        support=set(np.flatnonzero(A_true[i]>tol).tolist())
        support.discard(i)
        d=len(support)
        if not d: continue
        scores=A_hat[i].copy(); scores[i]=-np.inf
        pred=set(np.argsort(-scores,kind="mergesort")[:d].tolist())
        vals.append(len(support & pred)/d)
    return float(np.mean(vals))


def recovery_metrics(A_true,A_hat,v_true):
    diff=A_hat-A_true
    row_tv=0.5*np.abs(diff).sum(axis=1)
    support=A_true>1e-12
    support[np.eye(N,dtype=bool)]=False
    true_mass=np.mean([
        float(A_hat[i,support[i]].sum()) for i in range(N)
    ])
    v_hat=centrality_from_A(A_hat)

    eps=1e-12
    H=-(A_hat*np.log(np.clip(A_hat,eps,None))).sum(axis=1)
    H_norm=H/math.log(N-1)
    effective_neighbors=1.0/np.maximum((A_hat**2).sum(axis=1),eps)

    return {
        "A_MAE":float(np.mean(np.abs(diff))),
        "A_Fro":float(np.linalg.norm(diff,ord="fro")),
        "row_TV":float(np.mean(row_tv)),
        "true_edge_mass":float(true_mass),
        "support_topdegree_recall":support_topdegree_recall(A_true,A_hat),
        "edge_AP":edge_average_precision(A_true,A_hat),
        "centrality_L1":float(np.sum(np.abs(v_hat-v_true))),
        "row_entropy_norm":float(np.mean(H_norm)),
        "effective_neighbors":float(np.mean(effective_neighbors)),
        "A_max_row_mean":float(np.mean(A_hat.max(axis=1))),
    }


## Persistent sweep cache

In [ ]:
IDENTIFIER_PATH = REPO_ROOT / "opinion_dynamics" / "identify_nonlinear_unbounded_regularized.py"
HELPER_PATH = REPO_ROOT / "opinion_dynamics" / "experiments" / "online_single_shot_unbounded.py"

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

IMPLEMENTATION_HASH = hashlib.sha256(
    (sha256_file(IDENTIFIER_PATH)+sha256_file(HELPER_PATH)).encode()
).hexdigest()

SCIENTIFIC_CONFIG = {
    "pipeline_version":PIPELINE_VERSION,
    "source_study":SOURCE_STUDY,
    "source_config_hash":SOURCE_CONFIG_HASH,
    "N":N,"target":TARGET,"num_campaigns":NUM_CAMPAIGNS,
    "t_campaign":T_CAMPAIGN,"t_s":T_S,"max_u":MAX_U,
    "B_campaign":B_CAMPAIGN,"lambda_mix":LAMBDA_MIX,
    "fit_lr":FIT_LR,"fit_max_steps":FIT_MAX_STEPS,
    "fit_mae_stop":FIT_MAE_STOP,"fit_batch_size":FIT_BATCH_SIZE,
    "fit_check_every":FIT_CHECK_EVERY,
    "identifier_kwargs":IDENTIFIER_KWARGS,
    "learning_seed":LEARNING_SEED,
    "sweep":SWEEP,
}
CONFIG_HASH = hashlib.sha256(
    json.dumps(SCIENTIFIC_CONFIG,sort_keys=True).encode()
).hexdigest()[:16]

RUN_CACHE_ROOT = CACHE_ROOT / CONFIG_HASH
RUN_CACHE_ROOT.mkdir(parents=True,exist_ok=True)



def variant_id(kind,lam):
    return f"{kind}__lambda_{float(lam):.0e}".replace("+","")

def cache_paths_at(root, spec, kind, lam):
    d = root / spec["environment_id"]
    tag = variant_id(kind, lam)
    return d / f"{tag}.json", d / f"{tag}.npz"

def cache_paths(spec,kind,lam):
    return cache_paths_at(RUN_CACHE_ROOT, spec, kind, lam)

FIRST_SWEEP_CACHE_ROOT = CACHE_ROOT / FIRST_SWEEP_CONFIG_HASH
THETA_REFINEMENT_CACHE_ROOT = CACHE_ROOT / THETA_REFINEMENT_CONFIG_HASH

def _validate_cache_pair(jp, npz, spec, kind, lam, *, allowed_config_hashes):
    if not jp.exists() or not npz.exists():
        return None
    try:
        row = json.loads(jp.read_text(encoding="utf-8"))
    except Exception:
        return None

    if row.get("config_hash") not in allowed_config_hashes:
        return None
    if row.get("implementation_hash") != IMPLEMENTATION_HASH:
        return None
    if row.get("environment_id") != spec["environment_id"]:
        return None
    if row.get("regularizer_kind") != kind:
        return None
    if abs(float(row.get("regularizer_lambda", -1)) - float(lam)) >= 1e-18:
        return None
    return row

def _cache_hit(root, cfg_hash, spec, kind, lam, origin):
    jp, npz = cache_paths_at(root, spec, kind, lam)
    row = _validate_cache_pair(
        jp, npz, spec, kind, lam,
        allowed_config_hashes={cfg_hash},
    )
    if row is None:
        return None
    return {
        "row": row,
        "json_path": jp,
        "npz_path": npz,
        "origin": origin,
    }

def locate_cached_candidate(spec, kind, lam):
    # 1) Prefer candidates computed by this Gini refinement.
    hit = _cache_hit(
        RUN_CACHE_ROOT, CONFIG_HASH, spec, kind, lam, "current_gini_refinement"
    )
    if hit is not None:
        return hit

    # 2) The low-lambda theta-L2 sweep was computed in the second pass.
    if kind == "theta_l2":
        hit = _cache_hit(
            THETA_REFINEMENT_CACHE_ROOT,
            THETA_REFINEMENT_CONFIG_HASH,
            spec, kind, lam,
            "theta_refinement_reused",
        )
        if hit is not None:
            return hit

    # 3) Entropy and the four original Gini points came from the first sweep.
    if kind == "entropy" or (
        kind == "gini" and float(lam) in FIRST_SWEEP_GINI_LAMBDAS
    ):
        hit = _cache_hit(
            FIRST_SWEEP_CACHE_ROOT,
            FIRST_SWEEP_CONFIG_HASH,
            spec, kind, lam,
            "first_sweep_reused",
        )
        if hit is not None:
            return hit

    return None

def cache_valid(spec,kind,lam):
    return locate_cached_candidate(spec, kind, lam) is not None

def atomic_json(path,payload):
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(payload,indent=2,sort_keys=True,default=str),encoding="utf-8")
    os.replace(tmp,path)

def atomic_npz(path,**arrays):
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(".tmp")
    with tmp.open("wb") as h:
        np.savez_compressed(h,**arrays)
    os.replace(tmp,path)

def run_candidate(spec,kind,lam):
    cached = locate_cached_candidate(spec, kind, lam)
    if cached is not None:
        return cached["row"]

    # Only the five new Gini lambdas should normally arrive here.
    if kind != "gini" or float(lam) not in Gini_NEW_LAMBDAS:
        raise RuntimeError(
            f"Unexpected cache miss for {kind} lambda={lam}. "
            "Refusing to silently recompute a supposedly reusable condition."
        )

    X,Y,_=load_source_dataset(spec)
    model,A_hat,fit=fit_variant(X,Y,kind=kind,lam=lam)
    rollout=rollout_model(spec,model)
    rec=recovery_metrics(
        np.asarray(spec["A"],dtype=float),
        A_hat,
        np.asarray(spec["v_true"],dtype=float),
    )
    row={
        "config_hash":CONFIG_HASH,
        "implementation_hash":IMPLEMENTATION_HASH,
        "environment_id":spec["environment_id"],
        "environment_index":int(spec["environment_index"]),
        "scenario_class":spec["scenario_class"],
        "family":spec["family"],
        "dynamics":spec["dynamics"],
        "topology_seed":spec.get("topology_seed"),
        "opinion_seed":spec.get("opinion_seed"),
        "environment_seed":spec.get("environment_seed"),
        "learning_seed":LEARNING_SEED,
        "regularizer_kind":kind,
        "regularizer_lambda":float(lam),
        **fit,**rec,**rollout,
    }
    jp,npz=cache_paths(spec,kind,lam)
    atomic_npz(npz,A_hat=A_hat)
    atomic_json(jp,row)
    return row

print("Sweep config hash:",CONFIG_HASH)
print("Implementation hash:",IMPLEMENTATION_HASH[:16])
print("First-sweep cache:", FIRST_SWEEP_CACHE_ROOT)
print("Theta-refinement cache:", THETA_REFINEMENT_CACHE_ROOT)
print("New Gini lambdas:", sorted(Gini_NEW_LAMBDAS))


## Parallel sweep worker

In [ ]:
if WORKER_MODE:
    assigned=[
        spec for spec in ENVIRONMENT_SPECS
        if int(spec["environment_index"])%NUM_SHARDS==SHARD_ID
    ]
    print("Assigned environments:",len(assigned))
    print("Candidate fits:",len(assigned)*len(VARIANTS))
    count=0
    for pos,spec in enumerate(assigned,start=1):
        print(f"[{pos}/{len(assigned)}] {spec['environment_id']}",flush=True)
        for var in VARIANTS:
            run_candidate(
                spec,
                var["regularizer_kind"],
                var["regularizer_lambda"],
            )
            count+=1
    done=RESULTS_DIR/f"SHARD_{SHARD_ID}_COMPLETE.json"
    atomic_json(done,{
        "status":"success","shard_id":SHARD_ID,
        "n_environments":len(assigned),"n_candidates":count,
        "config_hash":CONFIG_HASH,"implementation_hash":IMPLEMENTATION_HASH,
    })
    print("Shard complete:",SHARD_ID)
else:
    print("Merge/analysis mode. Use the parallel launcher to perform the sweep.")


## Merge and analyze

The unregularized reference is loaded from the exact final September abundant
run (learning seed 0); it is **not retrained**.

The main tuning question is deliberately two-dimensional:

- does the regularizer improve recovery of the true weighted \(A\)?
- does it preserve or improve downstream control?

A candidate that merely produces a visually sparse matrix but harms control is
not considered successful.


In [ ]:
if not WORKER_MODE:
    missing=[
        (spec["environment_id"],v["regularizer_kind"],v["regularizer_lambda"])
        for spec in ENVIRONMENT_SPECS
        for v in VARIANTS
        if not cache_valid(spec,v["regularizer_kind"],v["regularizer_lambda"])
    ]
    print("Expected combined candidate evaluations:",len(ENVIRONMENT_SPECS)*len(VARIANTS))
    print("Expected newly fitted Gini candidates:", len(ENVIRONMENT_SPECS)*len(Gini_NEW_LAMBDAS))
    print("Missing:",len(missing))
    if missing:
        print("First missing:",missing[:10])
        raise RuntimeError("Sweep incomplete. Re-run the parallel launcher.")

    rows=[]
    cache_origins={}
    for spec in ENVIRONMENT_SPECS:
        for v in VARIANTS:
            cached=locate_cached_candidate(
                spec,v["regularizer_kind"],v["regularizer_lambda"]
            )
            if cached is None:
                raise RuntimeError(
                    f"Candidate vanished after validation: "
                    f"{spec['environment_id']} {v}"
                )
            row=dict(cached["row"])
            row["cache_origin"]=cached["origin"]
            rows.append(row)
            cache_origins[cached["origin"]]=cache_origins.get(cached["origin"],0)+1
    sweep_df=pd.DataFrame(rows)
    print("Cache origins:",cache_origins)

    # Exact no-regularization seed-0 reference.
    source_summary=pd.read_csv(SOURCE_RESULTS_DIR/"combined_summary.csv")
    no_reg=source_summary[
        (source_summary["policy"]=="learned_abundant_nonlinear")
        & (source_summary["learning_seed"]==LEARNING_SEED)
    ].copy()
    assert len(no_reg)==195

    # Add A-hat-derived recovery/sparsity metrics from final cached rollout files.
    ref_rows=[]
    for spec in ENVIRONMENT_SPECS:
        env_id=spec["environment_id"]
        src=SOURCE_CACHE_ROOT/env_id/f"learned_seed_{LEARNING_SEED:02d}_rollout.npz"
        with np.load(src) as arr:
            A_hat=np.asarray(arr["A_hat"],dtype=float)
        rec=recovery_metrics(
            np.asarray(spec["A"],dtype=float),A_hat,
            np.asarray(spec["v_true"],dtype=float),
        )
        base=no_reg[no_reg["environment_id"]==env_id].iloc[0]
        ref_rows.append({
            "environment_id":env_id,
            "environment_index":int(spec["environment_index"]),
            "scenario_class":spec["scenario_class"],
            "family":spec["family"],
            "dynamics":spec["dynamics"],
            "topology_seed":spec.get("topology_seed"),
            "opinion_seed":spec.get("opinion_seed"),
            "environment_seed":spec.get("environment_seed"),
            "learning_seed":LEARNING_SEED,
            "regularizer_kind":"none",
            "regularizer_lambda":0.0,
            "train_mae":float(base["train_mae"]),
            "identity_mae":float(base["identity_mae"]),
            "model_over_identity":float(base["model_over_identity"]),
            "mean_end":float(base["mean_end"]),
            "mean_over_trajectory":float(base["mean_over_trajectory"]),
            "final_mean_abs_distance_to_target":float(base["final_mean_abs_distance_to_target"]),
            **rec,
        })
    ref_df=pd.DataFrame(ref_rows)

    ALL=pd.concat([ref_df,sweep_df],ignore_index=True,sort=False)
    ALL.to_csv(RESULTS_DIR/"regularization_sweep_all_runs.csv",index=False)

    ref_map=ref_df.set_index("environment_id")
    for metric in [
        "row_TV","A_MAE","true_edge_mass","support_topdegree_recall",
        "edge_AP","centrality_L1","row_entropy_norm","effective_neighbors",
        "mean_end","model_over_identity",
    ]:
        ALL[f"delta_{metric}_vs_none"]=ALL.apply(
            lambda r: float(r[metric])-float(ref_map.loc[r["environment_id"],metric]),
            axis=1,
        )

    group_cols=["regularizer_kind","regularizer_lambda"]
    summary=ALL.groupby(group_cols,as_index=False).agg(
        n=("environment_id","size"),
        mean_end=("mean_end","mean"),
        model_over_identity=("model_over_identity","mean"),
        row_TV=("row_TV","mean"),
        true_edge_mass=("true_edge_mass","mean"),
        support_topdegree_recall=("support_topdegree_recall","mean"),
        edge_AP=("edge_AP","mean"),
        centrality_L1=("centrality_L1","mean"),
        row_entropy_norm=("row_entropy_norm","mean"),
        effective_neighbors=("effective_neighbors","mean"),
        delta_mean_end_vs_none=("delta_mean_end_vs_none","mean"),
        delta_row_TV_vs_none=("delta_row_TV_vs_none","mean"),
        delta_true_edge_mass_vs_none=("delta_true_edge_mass_vs_none","mean"),
        delta_edge_AP_vs_none=("delta_edge_AP_vs_none","mean"),
        delta_model_over_identity_vs_none=("delta_model_over_identity_vs_none","mean"),
    )
    summary.to_csv(RESULTS_DIR/"regularization_sweep_summary.csv",index=False)

    by_dyn=ALL.groupby(
        ["regularizer_kind","regularizer_lambda","scenario_class","dynamics"],
        as_index=False,
    ).agg(
        n=("environment_id","size"),
        mean_end=("mean_end","mean"),
        row_TV=("row_TV","mean"),
        true_edge_mass=("true_edge_mass","mean"),
        edge_AP=("edge_AP","mean"),
        centrality_L1=("centrality_L1","mean"),
        model_over_identity=("model_over_identity","mean"),
        delta_mean_end_vs_none=("delta_mean_end_vs_none","mean"),
        delta_row_TV_vs_none=("delta_row_TV_vs_none","mean"),
    )
    by_dyn.to_csv(RESULTS_DIR/"regularization_sweep_by_dynamics.csv",index=False)

    print("Overall sweep summary:")
    display(summary.sort_values(["regularizer_kind","regularizer_lambda"]).round(6))

    print("\nStructured HK specifically:")
    display(
        by_dyn[
            (by_dyn["scenario_class"]=="structured_mechanism")
            & (by_dyn["dynamics"]=="hegselmannkrause")
        ].sort_values(["regularizer_kind","regularizer_lambda"]).round(6)
    )

    # Pareto candidates: no other setting has both lower/equal row-TV and
    # higher/equal control, with one strict improvement.
    points=summary.copy()
    pareto=[]
    for i,r in points.iterrows():
        dominated=False
        for j,q in points.iterrows():
            if i==j: continue
            if (
                q["row_TV"] <= r["row_TV"]
                and q["mean_end"] >= r["mean_end"]
                and (
                    q["row_TV"] < r["row_TV"]
                    or q["mean_end"] > r["mean_end"]
                )
            ):
                dominated=True
                break
        if not dominated:
            pareto.append(i)
    pareto_df=points.loc[pareto].sort_values("row_TV")
    pareto_df.to_csv(RESULTS_DIR/"regularization_sweep_pareto.csv",index=False)

    print("\nPareto frontier (weighted-A recovery vs control):")
    display(
        pareto_df[
            ["regularizer_kind","regularizer_lambda","row_TV","mean_end",
             "true_edge_mass","edge_AP","model_over_identity",
             "effective_neighbors"]
        ].round(6)
    )

    # Plot: recovery vs control
    fig,ax=plt.subplots(figsize=(7,5))
    markers={"none":"o","theta_l2":"s","entropy":"^","gini":"D"}
    for kind,g in summary.groupby("regularizer_kind"):
        ax.scatter(
            g["row_TV"],g["mean_end"],
            marker=markers.get(kind,"o"),label=kind,s=55,
        )
        for _,r in g.iterrows():
            label="0" if kind=="none" else f"{r['regularizer_lambda']:.0e}"
            ax.annotate(label,(r["row_TV"],r["mean_end"]),xytext=(4,3),
                        textcoords="offset points",fontsize=8)
    ax.set_xlabel("Mean row-TV error of A (lower is better)")
    ax.set_ylabel("Final mean opinion (higher is better)")
    ax.set_title("Abundant regularization sweep: A recovery vs control")
    ax.legend()
    fig.tight_layout()
    fig.savefig(RESULTS_DIR/"regularization_sweep_recovery_vs_control.png",dpi=220)
    plt.show()

    fig,ax=plt.subplots(figsize=(7,5))
    for kind,g in summary.groupby("regularizer_kind"):
        ax.scatter(
            g["effective_neighbors"],g["row_TV"],
            marker=markers.get(kind,"o"),label=kind,s=55,
        )
        for _,r in g.iterrows():
            label="0" if kind=="none" else f"{r['regularizer_lambda']:.0e}"
            ax.annotate(label,(r["effective_neighbors"],r["row_TV"]),
                        xytext=(4,3),textcoords="offset points",fontsize=8)
    ax.set_xlabel("Mean effective neighbors per row (lower = more concentrated)")
    ax.set_ylabel("Mean row-TV error of A")
    ax.set_title("Concentration versus true weighted-A recovery")
    ax.legend()
    fig.tight_layout()
    fig.savefig(RESULTS_DIR/"regularization_sweep_concentration_vs_recovery.png",dpi=220)
    plt.show()

    atomic_json = lambda p,x: p.write_text(json.dumps(x,indent=2,default=str),encoding="utf-8")
    atomic_json(
        RESULTS_DIR/"COMPUTATION_COMPLETE.json",
        {
            "status":"success",
            "study_name":STUDY_NAME,
            "pipeline_version":PIPELINE_VERSION,
            "git_commit":GIT_COMMIT,
            "config_hash":CONFIG_HASH,
            "implementation_hash":IMPLEMENTATION_HASH,
            "source_study":SOURCE_STUDY,
            "source_config_hash":SOURCE_CONFIG_HASH,
            "n_environments":195,
            "learning_seed":LEARNING_SEED,
            "n_regularized_variants":len(VARIANTS),
            "n_candidate_evaluations":195*len(VARIANTS),
            "n_new_fits":195*len(Gini_NEW_LAMBDAS),
            "n_reused_fits":195*(
                len(SWEEP["theta_l2"])
                + len(SWEEP["entropy"])
                + len(FIRST_SWEEP_GINI_LAMBDAS)
            ),
            "first_sweep_config_hash":FIRST_SWEEP_CONFIG_HASH,
            "theta_refinement_config_hash":THETA_REFINEMENT_CONFIG_HASH,
            "gini_grid":[float(x) for x in SWEEP["gini"]],
            "completed_at_utc":datetime.now(timezone.utc).isoformat(),
        },
    )
    print("\nSweep analysis complete:",RESULTS_DIR)
